In [1]:
import s3fs
import os
import json
import mimetypes
import pandas as pd
import geopandas as gpd
from dbfread import DBF
import lasio
import fsspec
from io import StringIO, BytesIO
import boto3
from IPython.display import Image, display, IFrame
import matplotlib.pyplot as plt
%matplotlib inline
s3 = s3fs.S3FileSystem()

#### Look_up file

In [2]:
s3_client = boto3.client('s3')

s3_path = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/Lookup/ranger equipment lookup.csv"
bucket_name = s3_path.split('/')[2]  # Extract bucket name
file_key = "/".join(s3_path.split('/')[3:])  # Extract file key

response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
csv_data = response['Body'].read().decode('utf-8')  # Decode bytes to string

lookup_df = pd.read_csv(StringIO(csv_data))

lookup_df

,item_id,point_name,equipment,parent_equipment_name,equipment_type
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift
...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility


#### Reading History Folder under PatchIQ

In [ ]:
import boto3

s3_client = boto3.client('s3')

s3_path = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/History/"
bucket_name = s3_path.split('/')[2] 
prefix = "/".join(s3_path.split('/')[3:])

response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

if 'Contents' in response:
    file_paths = [f"s3://{bucket_name}/{obj['Key']}" for obj in response['Contents']]
    for file in file_paths:
        print(file)
else:
    print("No files found in the specified S3 folder.")


In [ ]:
file_paths
file_paths.pop(0)
file_paths

In [ ]:
cleaned_paths = [path.replace("s3://ayata-clients/", "") for path in file_paths]
cleaned_paths

#### Merge Data of all files in History folder of Patch_IQ

In [ ]:
# import boto3
# import pandas as pd
# from io import StringIO

# # Initialize S3 client
# s3_client = boto3.client('s3')

# # Define S3 bucket and folder path
# s3_folder = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/History/"
# bucket_name = s3_folder.split('/')[2]
# prefix = "/".join(s3_folder.split('/')[3:])  # Extract folder prefix

# # List all files in the folder
# response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

# # Extract file paths (excluding the folder itself)
# file_keys = [obj['Key'] for obj in response.get('Contents', []) if not obj['Key'].endswith('/')]

# # Define column names
# column_names = ["ID", "Timestamp", "Value"]

# # Create an empty list to store DataFrames
# dfs = []

# # Read each CSV file from S3 and append to the list
# for file_key in file_keys:
#     response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
#     csv_data = response['Body'].read().decode('utf-8') 
#     df = pd.read_csv(StringIO(csv_data), header=None, names=column_names, low_memory=False)
#     dfs.append(df)

# # Concatenate all DataFrames into one
# final_df = pd.concat(dfs, ignore_index=True)

# # Display the combined DataFrame
# # print(final_df.head())

# # Optionally, save the final DataFrame to a CSV file
# final_df.to_csv("merged_data.csv", index=False)

#### Reding The csv of History folder merged data of Patch_IQ

In [3]:
merged_data_df_Patch_IQ_History = pd.read_csv("merged_data.csv")
merged_data_df_Patch_IQ_History

,ID,Timestamp,Value
0,10034,2022-02-21 16:26:12.157+00,0.0
1,10093,2022-02-21 16:26:12.157+00,1.25
2,10080,2022-02-21 16:26:12.157+00,424.965
3,10022,2022-02-21 16:26:12.157+00,7.688
4,10102,2022-02-21 16:26:12.157+00,1.25
...,...,...,...
52611,10103,2022-02-28 17:08:18.999+00,111.597
52612,10094,2022-02-28 17:08:18.999+00,107.823
52613,10025,2022-02-28 17:08:18.999+00,9.591
52614,10101,2022-02-28 17:08:18.999+00,160.381


#### Joining of Patch_IQ History merged data with look up

In [4]:
result_df = merged_data_df_Patch_IQ_History.merge(lookup_df, left_on="ID", right_on="item_id", how="left")
result_df

,ID,Timestamp,Value,item_id,point_name,equipment,parent_equipment_name,equipment_type
0,10034,2022-02-21 16:26:12.157+00,0.0,10034.0,Tubing Pressure,HALITE B2H,HALITE Well Pad,Gas Lift
1,10093,2022-02-21 16:26:12.157+00,1.25,10093.0,Casing Pressure,TURQUOISE B 2H,TURQUOISE Well Pad,Gas Lift
2,10080,2022-02-21 16:26:12.157+00,424.965,10080.0,Tubing Pressure,SHARKTOOTH (SA) UNIT 1 1H,SHARKTOOTH Well Pad,Gas Lift
3,10022,2022-02-21 16:26:12.157+00,7.688,10022.0,Tubing Pressure,LAGER UNIT 4H,LAGER Well Pad,Gas Lift
4,10102,2022-02-21 16:26:12.157+00,1.25,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
52611,10103,2022-02-28 17:08:18.999+00,111.597,NaN,NaN,NaN,NaN,NaN
52612,10094,2022-02-28 17:08:18.999+00,107.823,10094.0,Tubing Pressure,TURQUOISE C 3H,TURQUOISE Well Pad,Gas Lift
52613,10025,2022-02-28 17:08:18.999+00,9.591,10025.0,Casing Pressure,DRAGONSTONE A 1H,HAWG 3H 4H 5H/DRAGONSTONE A1 B2 Well Pad,Gas Lift
52614,10101,2022-02-28 17:08:18.999+00,160.381,NaN,NaN,NaN,NaN,NaN


#### Saving the joining to a csv

In [5]:
result_df.to_csv('Patch_IQ_Look_Up_and_History.csv', index=False)

#### Operations on Patch_IQ_Look_Up_and_History joined data

In [ ]:
point_name_array = result_df["point_name"].unique()
print(point_name_array)

In [ ]:
equipment_array = result_df["equipment"].unique()
print(equipment_array)

In [ ]:
parent_equipment_name_array = result_df["parent_equipment_name"].unique()
print(parent_equipment_name_array)S

In [ ]:
equipment_type_name_array = result_df["equipment_type"].unique()
print(equipment_type_name_array)

In [ ]:
point_name_array = result_df["point_name"].unique()
print(point_name_array)

In [ ]:
tubing_pressure_df = result_df[result_df["point_name"] == "Tubing Pressure"]
tubing_pressure_df

In [ ]:
casing_pressure_df = result_df[result_df["point_name"] == "Casing Pressure"]
casing_pressure_df

#### Converting of all the files of History folders to csv

In [ ]:
# import boto3
# import pandas as pd
# import os
# import re
# from io import StringIO

# # Initialize S3 client
# s3_client = boto3.client('s3')

# # Define S3 bucket and folder path
# s3_folder = "s3://ayata-clients/Baytex_new/raw_data/PatchIQ/History/"
# bucket_name = s3_folder.split('/')[2]
# prefix = "/".join(s3_folder.split('/')[3:])  # Extract folder prefix

# # Local directory to save CSV files
# output_dir = "Patch_IQ_History"
# os.makedirs(output_dir, exist_ok=True)  # Create folder if not exists

# # List all files in the folder
# response = s3_client.list_objects_v2(Bucket=bucket_name, Prefix=prefix)

# # Extract file paths (excluding the folder itself)
# file_keys = [obj['Key'] for obj in response.get('Contents', []) if not obj['Key'].endswith('/')]

# # Define column names
# column_names = ["ID", "Timestamp", "Value"]

# # Process each file individually
# for file_key in file_keys:
#     response = s3_client.get_object(Bucket=bucket_name, Key=file_key)
#     csv_data = response['Body'].read().decode('utf-8')

#     # Read CSV data into a DataFrame
#     df = pd.read_csv(StringIO(csv_data), header=None, names=column_names, low_memory=False)

#     # Extract the file name (without folder path)
#     original_file_name = os.path.basename(file_key)

#     # Modify the filename by replacing dots in the timestamp with underscores
#     modified_file_name = re.sub(r'\.(\d{3})', r'_\1', original_file_name)

#     # Ensure it ends with .csv
#     if not modified_file_name.endswith(".csv"):
#         modified_file_name += ".csv"

#     # Save the file as CSV in the output folder
#     output_file = os.path.join(output_dir, modified_file_name)
#     df.to_csv(output_file, index=False)

#     print(f"Saved: {output_file}")

#### Uploding those files to S3

In [ ]:
# import boto3
# import os

# s3_client = boto3.client('s3')

# local_folder = "Patch_IQ_History"  # Replace with your folder path
# bucket_name = "ayata-clients"
# s3_folder = "Baytex_new/ayata_processed_data/Patch_IQ_History"  # S3 folder path

# def upload_folder_to_s3(local_folder, bucket_name, s3_folder):
#     for root, dirs, files in os.walk(local_folder):
#         for file in files:
#             local_path = os.path.join(root, file)
#             s3_path = os.path.join(s3_folder, os.path.relpath(local_path, local_folder))

#             # Replace backslashes with forward slashes for S3 compatibility
#             s3_path = s3_path.replace("\\", "/")

#             print(f"Uploading {local_path} to s3://{bucket_name}/{s3_path}")
#             s3_client.upload_file(local_path, bucket_name, s3_path)

# upload_folder_to_s3(local_folder, bucket_name, s3_folder)


#### Reading Iron_IQ file

In [6]:
import pandas as pd
iron_iQ = pd.read_excel('Iron IQ Well Name to Accounting ID Table.xlsx', engine='openpyxl')
iron_iQ

,Name,Accounting Number
0,ADDAX HUNTER #1H,203216.01
1,ADDAX HUNTER #2H,203217.01
2,ADDAX HUNTER #7H,204359.01
3,ADDAX HUNTER #8H,204360.01
4,ADDAX HUNTER 3H,203218.01
...,...,...
620,ZEBRA HUNTER #4H,207662.01
621,ZEBRA HUNTER #6H,204348.01
622,ZEBRA HUNTER #7H,204349.01
623,ZIRCON A1H,204755.01


#### Joining of look_up and Iron_IQ

In [7]:
outer_join_01 = pd.merge(lookup_df, iron_iQ, left_on='equipment', right_on='Name', how='left')
outer_join_01

,item_id,point_name,equipment,parent_equipment_name,equipment_type,Name,Accounting Number
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,204600.01
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,204600.01
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,204601.01
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,204601.01
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift,BLOODSTONE C 3H,204686.01
...,...,...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility,NaN,NaN
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility,NaN,NaN
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility,NaN,NaN
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility,NaN,NaN


#### Data without any NaN Value

In [8]:
no_nan_rows = outer_join_01[outer_join_01.notna().all(axis=1)]
no_nan_rows

,item_id,point_name,equipment,parent_equipment_name,equipment_type,Name,Accounting Number
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,204600.01
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,204600.01
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,204601.01
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,204601.01
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift,BLOODSTONE C 3H,204686.01
...,...,...,...,...,...,...,...
27864,62836,Static Pressure,TERRA B2H,TERRA Well Pad,Gas Lift,TERRA B2H,207612.01
27865,62837,Differential Pressure,TERRA C3H,TERRA Well Pad,Gas Lift,TERRA C3H,207613.01
27866,62838,Static Pressure,TERRA C3H,TERRA Well Pad,Gas Lift,TERRA C3H,207613.01
27867,62839,Differential Pressure,DEEDRA 1H,DEEDRA 1H Well Pad,Gas Lift,DEEDRA 1H,204425.01


#### Data Contain NaN Value

In [9]:
nan_rows = outer_join_01[outer_join_01.isna().any(axis=1)]
nan_rows

,item_id,point_name,equipment,parent_equipment_name,equipment_type,Name,Accounting Number
100,10114,Flow Rate,BIG FIVE A1 H Water Meter,BIG FIVE A1 H,Water Meter,NaN,NaN
101,10115,Flow Today,BIG FIVE A1 H Water Meter,BIG FIVE A1 H,Water Meter,NaN,NaN
102,10116,Flow Yesterday,BIG FIVE A1 H Water Meter,BIG FIVE A1 H,Water Meter,NaN,NaN
103,10117,Flow Rate,BIG FIVE A1 H Oil Meter,BIG FIVE A1 H,Oil Meter,NaN,NaN
104,10118,Flow Today,BIG FIVE A1 H Oil Meter,BIG FIVE A1 H,Oil Meter,NaN,NaN
...,...,...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility,NaN,NaN
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility,NaN,NaN
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility,NaN,NaN
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility,NaN,NaN


#### Accounting Number as NaN

In [10]:
name_nan_rows = outer_join_01[outer_join_01['Accounting Number'].isna()]
name_nan_rows

,item_id,point_name,equipment,parent_equipment_name,equipment_type,Name,Accounting Number
100,10114,Flow Rate,BIG FIVE A1 H Water Meter,BIG FIVE A1 H,Water Meter,NaN,NaN
101,10115,Flow Today,BIG FIVE A1 H Water Meter,BIG FIVE A1 H,Water Meter,NaN,NaN
102,10116,Flow Yesterday,BIG FIVE A1 H Water Meter,BIG FIVE A1 H,Water Meter,NaN,NaN
103,10117,Flow Rate,BIG FIVE A1 H Oil Meter,BIG FIVE A1 H,Oil Meter,NaN,NaN
104,10118,Flow Today,BIG FIVE A1 H Oil Meter,BIG FIVE A1 H,Oil Meter,NaN,NaN
...,...,...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility,NaN,NaN
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility,NaN,NaN
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility,NaN,NaN
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility,NaN,NaN


#### Adding new column as Well Name which is basically a copy of the equipment in the lookup file

In [11]:
lookup_df['well_name'] = lookup_df['equipment']
lookup_df

,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift,BLOODSTONE C 3H
...,...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility,PECAN Facility
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility,MATOCHA 1H Facility
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility,COLLEEN CALYPSO Facility
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility,HAWKEYE 9H 10H 11H Facility


In [12]:
import re

unwanted_strings = [
    'Water Meter', 'Oil Meter', 'Oil Tank 1', 'Gas Lift Meter', 'Water Tank'
    'Production Meter', 'Reject Tank', 'Oil Tank 2', 'Oil Tank 3', 'Water Tank #1', 'Water Tank #2',
    'Water Tank #3', 'Water Tank #4', 'Water Tank #5', 'Water Tank #6', 'Oil(Reject) Tank #5'
    'Oil Tank 4', 'Oil Tank 5', 'Meter', 'HP Meter', 'Jet Pump', 'Gas Lift', 'Drip Tank', 'Lact'
    'Remote Device', 'Oil Tank', 'Water Tank 1', 'Water Tank 2', 'Water Tank 3'
    'Cooler Scrubber', 'Separator', 'Fuel Scrubber', 'Oil(Reject) Tank #5'
    'Flare Scrubber', 'Facility', 'Flash Gas', 'Tank Battery', 'Heater Separator',
    'Heater Treater', 'Flare', 'Production', 'Water Test Tank', 'Water Common Tank',
    'Oil Test Tank', 'Oil Common Tank', 'Seperator', 'Bulk', 'Tank 1', 'Tank 2',
    'Oil Common Tank', 'Water Test Tank', 'Common', 'Production', 'Injection',
    'Compressor Discharge', 'Compressor Fuel Gas', 'Sales Check', 'Sales', 'Robin',
    'Unknown', 'Unknown Tank 3', 'Unknown Tank 4', 'Unknown Tank 5', 'Unknown Tank 6',
    'Unknown Tank 7', 'Unknown Tank 8', 'Water Tank', 'Remove Device', 'Cooler Scrubber', 'Scrubber',
    'Fuel', 'Tank', 'Water Tank #1', 'Water Tank #2', '#1', '#2', '#3', '#4', '#6', '#5', 'Gas Lift'
]

pattern = '|'.join(map(re.escape, unwanted_strings))
lookup_df['well_name'] = lookup_df['well_name'].str.replace(pattern, '', regex=True).str.strip()
lookup_df

,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift,BLOODSTONE C 3H
...,...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility,PECAN
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility,MATOCHA 1H
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility,COLLEEN CALYPSO
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility,HAWKEYE 9H 10H 11H


In [13]:
join_well_name = pd.merge(lookup_df, iron_iQ, left_on='equipment', right_on='Name', how='left')
join_well_name

,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name,Name,Accounting Number
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,BIG FIVE A1 H,204600.01
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,BIG FIVE A1 H,204600.01
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,BIG FIVE B 2H,204601.01
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,BIG FIVE B 2H,204601.01
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift,BLOODSTONE C 3H,BLOODSTONE C 3H,204686.01
...,...,...,...,...,...,...,...,...
28454,71106,PLC Battery Voltage,PECAN Facility,NaN,Facility,PECAN,NaN,NaN
28455,71107,PLC Battery Voltage,MATOCHA 1H Facility,NaN,Facility,MATOCHA 1H,NaN,NaN
28456,71108,PLC Battery Voltage,COLLEEN CALYPSO Facility,NaN,Facility,COLLEEN CALYPSO,NaN,NaN
28457,71109,PLC Battery Voltage,HAWKEYE 9H 10H 11H Facility,NaN,Facility,HAWKEYE 9H 10H 11H,NaN,NaN


In [14]:
no_nan_rows = join_well_name[join_well_name.notna().all(axis=1)]
no_nan_rows

,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name,Name,Accounting Number
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,BIG FIVE A1 H,204600.01
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,BIG FIVE A1 H,204600.01
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,BIG FIVE B 2H,204601.01
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,BIG FIVE B 2H,204601.01
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift,BLOODSTONE C 3H,BLOODSTONE C 3H,204686.01
...,...,...,...,...,...,...,...,...
27864,62836,Static Pressure,TERRA B2H,TERRA Well Pad,Gas Lift,TERRA B2H,TERRA B2H,207612.01
27865,62837,Differential Pressure,TERRA C3H,TERRA Well Pad,Gas Lift,TERRA C3H,TERRA C3H,207613.01
27866,62838,Static Pressure,TERRA C3H,TERRA Well Pad,Gas Lift,TERRA C3H,TERRA C3H,207613.01
27867,62839,Differential Pressure,DEEDRA 1H,DEEDRA 1H Well Pad,Gas Lift,DEEDRA 1H,DEEDRA 1H,204425.01


In [15]:
no_nan_rows.to_csv('joined_updated_iron_iQ.csv', index=False)

In [17]:
import pandas as pd
csv_of_merged_data_df_Patch_IQ_History = pd.read_csv('merged_data.csv')
csv_of_merged_data_df_Patch_IQ_History

,ID,Timestamp,Value
0,10034,2022-02-21 16:26:12.157+00,0.0
1,10093,2022-02-21 16:26:12.157+00,1.25
2,10080,2022-02-21 16:26:12.157+00,424.965
3,10022,2022-02-21 16:26:12.157+00,7.688
4,10102,2022-02-21 16:26:12.157+00,1.25
...,...,...,...
52611,10103,2022-02-28 17:08:18.999+00,111.597
52612,10094,2022-02-28 17:08:18.999+00,107.823
52613,10025,2022-02-28 17:08:18.999+00,9.591
52614,10101,2022-02-28 17:08:18.999+00,160.381


In [18]:
csv_of_merged_data_df_Patch_IQ_History_Look_up_Iron_IQ = pd.read_csv('joined_updated_iron_iQ.csv')
csv_of_merged_data_df_Patch_IQ_History_Look_up_Iron_IQ

,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name,Name,Accounting Number
0,10000,Tubing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,BIG FIVE A1 H,204600.01
1,10001,Casing Pressure,BIG FIVE A1 H,BIG FIVE Well Pad,Gas Lift,BIG FIVE A1 H,BIG FIVE A1 H,204600.01
2,10002,Tubing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,BIG FIVE B 2H,204601.01
3,10003,Casing Pressure,BIG FIVE B 2H,BIG FIVE Well Pad,Gas Lift,BIG FIVE B 2H,BIG FIVE B 2H,204601.01
4,10004,Tubing Pressure,BLOODSTONE C 3H,BLOODSTONE C 3H D 4H Well Pad,Gas Lift,BLOODSTONE C 3H,BLOODSTONE C 3H,204686.01
...,...,...,...,...,...,...,...,...
2613,62836,Static Pressure,TERRA B2H,TERRA Well Pad,Gas Lift,TERRA B2H,TERRA B2H,207612.01
2614,62837,Differential Pressure,TERRA C3H,TERRA Well Pad,Gas Lift,TERRA C3H,TERRA C3H,207613.01
2615,62838,Static Pressure,TERRA C3H,TERRA Well Pad,Gas Lift,TERRA C3H,TERRA C3H,207613.01
2616,62839,Differential Pressure,DEEDRA 1H,DEEDRA 1H Well Pad,Gas Lift,DEEDRA 1H,DEEDRA 1H,204425.01


In [19]:
final_join = pd.merge(csv_of_merged_data_df_Patch_IQ_History, csv_of_merged_data_df_Patch_IQ_History_Look_up_Iron_IQ, 
                          left_on='ID', right_on='item_id', how='left')
final_join

,ID,Timestamp,Value,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name,Name,Accounting Number
0,10034,2022-02-21 16:26:12.157+00,0.0,10034.0,Tubing Pressure,HALITE B2H,HALITE Well Pad,Gas Lift,HALITE B2H,HALITE B2H,204728.01
1,10093,2022-02-21 16:26:12.157+00,1.25,10093.0,Casing Pressure,TURQUOISE B 2H,TURQUOISE Well Pad,Gas Lift,TURQUOISE B 2H,TURQUOISE B 2H,204701.01
2,10080,2022-02-21 16:26:12.157+00,424.965,10080.0,Tubing Pressure,SHARKTOOTH (SA) UNIT 1 1H,SHARKTOOTH Well Pad,Gas Lift,SHARKTOOTH (SA) UNIT 1 1H,SHARKTOOTH (SA) UNIT 1 1H,204511.01
3,10022,2022-02-21 16:26:12.157+00,7.688,10022.0,Tubing Pressure,LAGER UNIT 4H,LAGER Well Pad,Gas Lift,LAGER UNIT 4H,LAGER UNIT 4H,204309.01
4,10102,2022-02-21 16:26:12.157+00,1.25,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
52611,10103,2022-02-28 17:08:18.999+00,111.597,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
52612,10094,2022-02-28 17:08:18.999+00,107.823,10094.0,Tubing Pressure,TURQUOISE C 3H,TURQUOISE Well Pad,Gas Lift,TURQUOISE C 3H,TURQUOISE C 3H,204717.01
52613,10025,2022-02-28 17:08:18.999+00,9.591,10025.0,Casing Pressure,DRAGONSTONE A 1H,HAWG 3H 4H 5H/DRAGONSTONE A1 B2 Well Pad,Gas Lift,DRAGONSTONE A 1H,DRAGONSTONE A 1H,204683.01
52614,10101,2022-02-28 17:08:18.999+00,160.381,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
final_join_no_nan_rows = final_join[final_join.notna().all(axis=1)]
final_join_no_nan_rows

,ID,Timestamp,Value,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name,Name,Accounting Number
0,10034,2022-02-21 16:26:12.157+00,0.0,10034.0,Tubing Pressure,HALITE B2H,HALITE Well Pad,Gas Lift,HALITE B2H,HALITE B2H,204728.01
1,10093,2022-02-21 16:26:12.157+00,1.25,10093.0,Casing Pressure,TURQUOISE B 2H,TURQUOISE Well Pad,Gas Lift,TURQUOISE B 2H,TURQUOISE B 2H,204701.01
2,10080,2022-02-21 16:26:12.157+00,424.965,10080.0,Tubing Pressure,SHARKTOOTH (SA) UNIT 1 1H,SHARKTOOTH Well Pad,Gas Lift,SHARKTOOTH (SA) UNIT 1 1H,SHARKTOOTH (SA) UNIT 1 1H,204511.01
3,10022,2022-02-21 16:26:12.157+00,7.688,10022.0,Tubing Pressure,LAGER UNIT 4H,LAGER Well Pad,Gas Lift,LAGER UNIT 4H,LAGER UNIT 4H,204309.01
5,10085,2022-02-21 16:26:12.157+00,0.5,10085.0,Casing Pressure,TOPAZ A 1H,TOPAZ Well Pad,Gas Lift,TOPAZ A 1H,TOPAZ A 1H,204714.01
...,...,...,...,...,...,...,...,...,...,...,...
52609,10081,2022-02-28 17:08:18.999+00,57093.992,10081.0,Casing Pressure,SHARKTOOTH (SA) UNIT 1 1H,SHARKTOOTH Well Pad,Gas Lift,SHARKTOOTH (SA) UNIT 1 1H,SHARKTOOTH (SA) UNIT 1 1H,204511.01
52610,10096,2022-02-28 17:08:18.999+00,87.363,10096.0,Tubing Pressure,PLATYPUS HUNTER #2H,HALITE Well Pad,Gas Lift,PLATYPUS HUNTER H,PLATYPUS HUNTER #2H,204220.01
52612,10094,2022-02-28 17:08:18.999+00,107.823,10094.0,Tubing Pressure,TURQUOISE C 3H,TURQUOISE Well Pad,Gas Lift,TURQUOISE C 3H,TURQUOISE C 3H,204717.01
52613,10025,2022-02-28 17:08:18.999+00,9.591,10025.0,Casing Pressure,DRAGONSTONE A 1H,HAWG 3H 4H 5H/DRAGONSTONE A1 B2 Well Pad,Gas Lift,DRAGONSTONE A 1H,DRAGONSTONE A 1H,204683.01


In [23]:
# Convert 'Timestamp' to datetime format for sorting
final_join_no_nan_rows['Timestamp'] = pd.to_datetime(final_join_no_nan_rows['Timestamp'], format='mixed')

# Sort data by 'Timestamp' in descending order and keep the most recent entry for each equipment
latest_data = final_join_no_nan_rows.sort_values('Timestamp', ascending=False).drop_duplicates('equipment')
latest_data

C:\Users\soura\AppData\Local\Temp\ipykernel_13968\670473130.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_join_no_nan_rows['Timestamp'] = pd.to_datetime(final_join_no_nan_rows['Timestamp'], format='mixed')


,ID,Timestamp,Value,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name,Name,Accounting Number
26106,10054,2022-03-07 16:38:53.630000+00:00,191.43,10054.0,Tubing Pressure,EFFENBERGER #4H,EFFENBERGER Well Pad,Gas Lift,EFFENBERGER H,EFFENBERGER #4H,203303.01
26107,10070,2022-03-07 16:38:53.630000+00:00,0,10070.0,Tubing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
26108,10069,2022-03-07 16:38:53.630000+00:00,450,10069.0,Casing Pressure,QUARTZ 1H,QUARTZ Well Pad,Gas Lift,QUARTZ 1H,QUARTZ 1H,204712.01
26109,10056,2022-03-07 16:38:53.630000+00:00,25.52,10056.0,Tubing Pressure,EFFENBERGER #5H,EFFENBERGER Well Pad,Gas Lift,EFFENBERGER H,EFFENBERGER #5H,203302.01
26128,10046,2022-03-07 16:38:53.629000+00:00,141.84,10046.0,Tubing Pressure,MAMMOTH A 1H,MAMMOTH Well Pad,Gas Lift,MAMMOTH A 1H,MAMMOTH A 1H,204608.01
26121,10044,2022-03-07 16:38:53.629000+00:00,32.29,10044.0,Tubing Pressure,LABRADORITE C 3H,LABRADORITE Well Pad,Gas Lift,LABRADORITE C 3H,LABRADORITE C 3H,204698.01
26129,10062,2022-03-07 16:38:53.629000+00:00,0.001,10062.0,Tubing Pressure,ROCK CREEK #11H,ROCK CREEK RANCH Well Pad,Gas Lift,ROCK CREEK 1H,ROCK CREEK #11H,203139.01
26127,10039,2022-03-07 16:38:53.629000+00:00,8,10039.0,Casing Pressure,PYRITE 1H,PYRITE Well Pad,Gas Lift,PYRITE 1H,PYRITE 1H,204711.01
26126,10041,2022-03-07 16:38:53.629000+00:00,8,10041.0,Casing Pressure,LABRADORITE A 1H,LABRADORITE Well Pad,Gas Lift,LABRADORITE A 1H,LABRADORITE A 1H,204696.01
26125,10059,2022-03-07 16:38:53.629000+00:00,23.678,10059.0,Tubing Pressure,RATTLER B 2H,RATTLER Well Pad,Gas Lift,RATTLER B 2H,RATTLER B 2H,204594.01


In [24]:
filtered_data = final_join_no_nan_rows[final_join_no_nan_rows['equipment'].isin(['SAPPHIRE A1H'])]
filtered_data

,ID,Timestamp,Value,item_id,point_name,equipment,parent_equipment_name,equipment_type,well_name,Name,Accounting Number
51,10072,2022-02-21 16:26:11.139000+00:00,25.465,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
101,10072,2022-02-21 15:42:18.704000+00:00,12.51,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
194,10072,2022-02-21 15:26:19.227000+00:00,7.789,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
204,10070,2022-02-21 14:56:07.060000+00:00,0.0,10070.0,Tubing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
281,10072,2022-02-21 13:56:09.113000+00:00,406.176,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
...,...,...,...,...,...,...,...,...,...,...,...
52373,10072,2022-02-28 18:24:08.376000+00:00,60.245,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
52435,10072,2022-02-28 18:07:30.741000+00:00,55.36,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
52466,10072,2022-02-28 17:52:31.475000+00:00,50.949,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01
52530,10072,2022-02-28 17:22:31.552000+00:00,42.072,10072.0,Casing Pressure,SAPPHIRE A1H,SAPPHIRE Well Pad,Gas Lift,SAPPHIRE A1H,SAPPHIRE A1H,204688.01


#### Ploting SAPPHIRE A1H Well